In [ ]:

<!DOCTYPE html>
<html>
<head>
  <meta charset="UTF-8">
  <title>University Rankings Heatmap</title>

  <script src="https://cdn.jsdelivr.net/npm/papaparse@5.4.1/papaparse.min.js"></script>
  <script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
</head>

<body>
  <h2>University Rankings (2016–2027)</h2>

  <input type="file" id="csvFile" accept=".csv">
  <div id="heatmap"></div>

  <script>
    const years = [
      "2016", "2017", "2018", "2019",
      "2020", "2021", "2022", "2023",
      "2024", "2025", "2026", "2027"
    ];

    document.getElementById("csvFile")
      .addEventListener("change", function(event) {

      const file = event.target.files[0];
      if (!file) return;

      Papa.parse(file, {
        header: true,
        skipEmptyLines: true,
        complete: function(results) {

          // Convert rankings to numbers
          const rankings = results.data.map(row => {
            const university = {
              name: row["University Name"],
              ipeds: row["IPEDS"]
            };

            years.forEach(year => {
              const value = row[year];
              university[year] =
                value !== undefined &&
                value !== null &&
                value.trim() !== ""
                  ? Number(value)
                  : null;

              if (!Number.isFinite(university[year])) {
                university[year] = null;
              }
            });

            return university;
          });

          // Select top 50 universities in 2016
          const top50 = rankings
            .filter(row =>
              row["2016"] !== null &&
              row["2016"] <= 50
            )
            .sort((a, b) => a["2016"] - b["2016"]);

          const universities = top50.map(row => row.name);

          // Heatmap numerical data
          // Rank >50 or missing: no blue fill
          const z = top50.map(row =>
            years.map(year => {
              const value = row[year];
              return value !== null && value <= 50
                ? value
                : null;
            })
          );

          // Text labels
          const labels = top50.map(row =>
            years.map(year => {
              const value = row[year];

              if (value === null) return "";
              if (value > 50) return ">50";
              return String(Math.trunc(value));
            })
          );

          // Background gray heatmap
          const background = {
            z: top50.map(row =>
              years.map(() => 1)
            ),
            x: years,
            y: universities,
            type: "heatmap",
            colorscale: [
              [0, "lightgrey"],
              [1, "lightgrey"]
            ],
            showscale: false,
            hoverinfo: "skip"
          };

          // Blue ranking heatmap
          const heatmap = {
            z: z,
            x: years,
            y: universities,
            type: "heatmap",
            colorscale: "Blues",
            reversescale: true,
            zmin: 1,
            zmax: 50,
            text: labels,
            texttemplate: "%{text}",
            textfont: {
              size: 10,
              color: "black"
            },
            hovertemplate:
              "%{y}<br>Year: %{x}" +
              "<br>Rank: %{z}<extra></extra>",
            colorbar: {
              title: "Rank",
              tickvals: [1, 10, 20, 30, 40, 50],
              autorange: "reversed"
            },
            xgap: 1,
            ygap: 1
          };

          // Add labels for gray cells
          const annotations = [];

          top50.forEach((row, i) => {
            years.forEach(year => {
              if (row[year] !== null && row[year] > 50) {
                annotations.push({
                  x: year,
                  y: universities[i],
                  text: "&gt;50",
                  showarrow: false,
                  font: {
                    size: 10,
                    color: "black"
                  }
                });
              }
            });
          });

          const layout = {
            title: "University Rankings (2016–2027)",
            width: 1100,
            height: Math.max(800, universities.length * 28),
            margin: {
              l: 290,
              r: 100,
              t: 70,
              b: 80
            },
            xaxis: {
              title: "Year",
              tickangle: -45,
              side: "bottom"
            },
            yaxis: {
              title: "University",
              autorange: "reversed",
              type: "category",
              categoryorder: "array",
              categoryarray: universities
            },
            annotations: annotations,
            plot_bgcolor: "lightgrey"
          };

          Plotly.newPlot(
            "heatmap",
            [background, heatmap],
            layout
          );
        }
      });
    });
  </script>
</body>
</html>

}
